In [5]:
import pandas as pd
from datetime import datetime
import os

df_init = pd.read_excel(r"C:\Users\Admin\OneDrive\Desktop\data science project\cfm-nlp\cfm-nlp\Automation.xlsx", sheet_name='IIP edit')

path_dict = dict()
list_1 = ['IMF IIP Annual.xlsx', 'IMF IIP Quarterly.xlsx']
# directory_path= r'C:\Users\Admin\OneDrive - The SOUTH-EAST ASIAN CENTRAL BANKS (SEACEN) RESEARCH AND TRAINING\Roger and Aizudeen\Country BoP and IIP Data'
directory_path= r'C:\Users\Admin\OneDrive - The SOUTH-EAST ASIAN CENTRAL BANKS (SEACEN) RESEARCH AND TRAINING\Roger and Aizudeen\Country BoP and IIP Data'
for filename in os.listdir(directory_path):
    if filename in ['01 Emerging Market', '02 G7 Countries']:
        continue
    file_path = os.path.join(directory_path, filename)
    list_temp = []
    for filename2 in os.listdir(file_path):
        if any(word in str(filename2) for word in list_1):            
            list_temp.append(filename2)
            path_dict[filename] = list_temp
        else:
            continue

df_path = []
for k, v in path_dict.items():
    df_path.append(os.path.join(directory_path, k, v[1]))
    # print(v[1]) ## to check the file name

df_all_annual = pd.DataFrame()
for x, y in enumerate(df_path):
    df = pd.read_excel(y)
    df = df[df[df.columns[0]].isin(df_init['desc'])]
    df_all_annual = pd.concat([df_all_annual, df]) 

df_all_annual.rename(columns={df_all_annual.columns[0]: 'Type'}, inplace=True)

df_all_annual_2 = df_all_annual[df_all_annual.columns[:4].to_list() + [dt for dt in df_all_annual.columns[4:].sort_values(ascending=True).to_list() if dt >= datetime(2019, 1, 1)]]

df_init_tw = pd.read_excel(r"C:\Users\Admin\OneDrive - The SOUTH-EAST ASIAN CENTRAL BANKS (SEACEN) RESEARCH AND TRAINING\Desktop\NLP Project\cfm-nlp\Automation.xlsx", sheet_name='Taiwan IIP')
df_tw = pd.read_excel(r"C:\Users\Admin\OneDrive - The SOUTH-EAST ASIAN CENTRAL BANKS (SEACEN) RESEARCH AND TRAINING\Roger and Aizudeen\Country BoP and IIP Data\Chinese Taipei (Taiwan) (TW)\TW CB IIP Annual.xlsx")
df_tw.rename(columns={df_tw.columns[0]: 'Type'}, inplace=True)
df_tw = df_tw[df_tw['Type'].isin(df_init_tw['desc'])]

for x, y in enumerate(df_tw['Type'].to_list()):
    df_tw.iat[x,0] = list(df_init_tw[(df_init_tw['desc'] == y)]['desc2'])[0]

selected_columns = [col for col in df_tw.columns[4:] if pd.to_datetime(col) >= datetime(2019, 1, 1)]
final_columns = list(df_tw.columns[:4]) + selected_columns
df_tw = df_tw[final_columns]

####baru
tw_col1 = df_all_annual_2.columns[4:][~df_all_annual_2.columns[4:].isin(df_tw.columns[4:])]
tw_col2 = {x.year:x for x in df_tw.columns[4:]}
df_tw2 = df_tw.copy()

for x in tw_col1:
    year = x.year
    if year in list(tw_col2.keys()):
        df_tw2[x] = df_tw2[tw_col2[year]]
    else:
        df_tw2[x] = None
####baru

df_all_annual_2 = pd.concat([df_all_annual_2, df_tw2], axis=0)

seacen_country = ['Papua New Guinea', 'Vietnam', 'Nepal', 'India', 'Indonesia', 'Laos', 'Sri Lanka', 'Hong Kong SAR (China)', 'Philippines', 'Taiwan', 'Malaysia', 'Mongolia', 'China', 'Cambodia', 'Thailand', 'Singapore', 'South Korea', 'Brunei', 'Myanmar']

for x in df_all_annual_2['Type'].values:
    var1 = list(set(df_all_annual_2[(df_all_annual_2['Type'] == x)]['Region'].values).symmetric_difference(set(seacen_country)))
    if len(var1) > 0:
        for y in var1:
            df_temp1 = pd.DataFrame({"Type": [x], "Region": [y]})
            df_all_annual_2 = pd.concat([df_all_annual_2, df_temp1], axis=0)
    else:
        continue

df_all_annual_2.sort_values(by=['Type', 'Region'], inplace=True)
df_all_annual_2.reset_index(drop=True, inplace=True)

init_dict = df_init.to_dict()
df_dict = dict()
for x in init_dict['desc'].values():
    df_temp = df_all_annual_2[(df_all_annual_2[df_all_annual_2.columns[0]] == x)].reset_index(drop=True)
    df_dict[x] = df_temp

var1 = {x*3:x for x in range(1,5)}
var2 = [f'{var1[x.month]}Q{x.year}' for x in df_all_annual_2.columns[4:].to_list()]
var3 = [x for x in var2 if x[:2] in ['2Q', '4Q']]
var4 = {'2Q':'1H', '4Q': '2H'}
var5 = [f'{var4[x[:2]]}{x[2:]}' for x in var3]

df_temp_col_1 = df_all_annual_2.rename(columns={k:v for k, v in zip(df_all_annual_2.columns[4:].to_list(), var2)})
df_temp_col_1 = df_temp_col_1[[*df_temp_col_1.columns[:4], *var3]]
df_temp_col_1.rename(columns={k:v for k,v in zip(var3, var5)}, inplace= True)
df_temp_col_1[df_temp_col_1.columns[4:]] = df_temp_col_1[df_temp_col_1.columns[4:]]/1000

df_temp_col_1.drop(columns=['Last Update Time', 'Unit'], inplace=True)

calc_dict = {'Asia-Pacific': seacen_country,
            'Asian Advance Economies': ['Hong Kong SAR (China)', 'South Korea', 'Singapore', 'Taiwan'],
            'ASEAN5 Economies': ['Indonesia', 'Malaysia', 'Philippines', 'Thailand', 'Vietnam'],
            'Asian EDMEs': ['Brunei', 'Cambodia', 'Laos', 'Mongolia', 'Nepal', 'Sri Lanka', 'Papua New Guinea', 'Myanmar'],
            'China': ['China'],
            'India': ['India'],
            }

C:\Users\Admin\AppData\Local\Temp\ipykernel_13864\2783400462.py:63: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_all_annual_2 = pd.concat([df_all_annual_2, df_tw2], axis=0)


In [20]:
###################### this is for equity ######################

asset_row = df_init.iloc[[0,4,5,6,8,9,10,11,12,13,14]]['desc'].to_list()
asset_row2 = df_init.iloc[[0,4,5,6,8,9,10,11,12,13,14]]['short_title'].to_list()
sum_rows = df_init[(df_init['short_title'].isin([asset_row2[4], asset_row2[7], asset_row2[9]]))]['desc'].to_list()
order = ['Direct Investment','Portfolio Equity','Portfolio Debt','Financial Derivatives','Currency & Deposits','Loans','Trade Credit & Advances','Other Investments','Reserve Assets',]
df_by_region_main = pd.DataFrame(columns=df_temp_col_1.columns)

for x,y in calc_dict.items():
    # print(x,y)
    df_by_region_temp = df_temp_col_1[(df_temp_col_1['Region'].isin(y)) & (df_temp_col_1['Type'].isin(asset_row))]
    df_by_region_temp = df_by_region_temp.groupby(['Type']).sum()
    df_by_region_temp.loc[str(x) + ' Total'] = df_by_region_temp.sum()
    df_by_region_temp.loc['Other Investments'] = df_by_region_temp.loc[sum_rows].sum()
    df_by_region_temp.reset_index(inplace = True)
    df_by_region_temp = df_by_region_temp[(~df_by_region_temp['Type'].isin(sum_rows))] #####  the symbol of '~' use similar like notin, the oposite of isin. this is build in python operator
    df_by_region_temp['Type'] = df_by_region_temp['Type'].replace(asset_row, asset_row2)
    df_by_region_temp = df_by_region_temp.sort_values(by='Type', key=lambda x: x.map({k:i for i, k in enumerate(order)}))
    df_by_region_temp['Region'] = x

    df_by_region_main = pd.concat([df_by_region_main, df_by_region_temp], axis = 0)  


df_by_region_main.reset_index(inplace = True, drop=True)
df_by_region_main.insert(loc=0, column='Group', value='Assets')

###################### this is for equity ######################
# 0,4,5,6,8,9,10,11,12,13,14

# df_by_region_main[(df_by_region_main['Type'] == 'Other Investments')]
df_by_region_main['Type'].unique()
# 4, 7, 9

C:\Users\Admin\AppData\Local\Temp\ipykernel_13864\2175195883.py:21: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_by_region_main = pd.concat([df_by_region_main, df_by_region_temp], axis = 0)


array(['Other Investments', 'FDI Assets', 'Financial Derivative Assets',
       'Official Reserve Assets', 'Currency and Deposits Assets',
       'Loans Assets', 'Trade Credits and Advances Assets',
       'Portfolio Debt Assets', 'Portfolio Equity Assets',
       'Asia-Pacific Total', 'Asian Advance Economies Total',
       'ASEAN5 Economies Total', 'Asian EDMEs Total', 'China Total',
       'India Total'], dtype=object)

In [14]:
df_init2 = pd.read_excel(r"C:\Users\Admin\OneDrive\Desktop\data science project\cfm-nlp\cfm-nlp\Automation.xlsx", sheet_name='BoP edit')
df_init2[(df_init2['short_title'] == 'Other Investments')]

,group,short_title,desc


In [17]:
###################### this is for debt ######################

debt_row = df_init.iloc[[16,20,21,22,24,25,26,27,28,29,30]]['desc'].to_list()
debt_row2 = df_init.iloc[[16,20,21,22,24,25,26,27,28,29,30]]['short_title'].to_list()
sum_rows2 = df_init[(df_init['short_title'].isin(df_init.iloc[[24,27,29,30]]['short_title'].to_list()))]['desc'].to_list()
order2 = ['Direct Investment','Portfolio Equity','Portfolio Debt','Financial Derivatives','Currency & Deposits','Loans','Trade Credit & Advances','Other Investments']
df_by_region_main2 = pd.DataFrame(columns=df_temp_col_1.columns)

for x,y in calc_dict.items():
    # print(x,y)
    df_by_region_temp = df_temp_col_1[(df_temp_col_1['Region'].isin(y)) & (df_temp_col_1['Type'].isin(debt_row))]
    df_by_region_temp = df_by_region_temp.groupby(['Type']).sum()
    df_by_region_temp.loc[str(x) + ' Total'] = df_by_region_temp.sum()
    df_by_region_temp.loc['Other Investments'] = df_by_region_temp.loc[sum_rows2].sum()
    df_by_region_temp.reset_index(inplace = True)
    df_by_region_temp = df_by_region_temp[(~df_by_region_temp['Type'].isin(sum_rows2))] #####  the symbol of '~' use similar like notin, the oposite of isin. this is build in python operator
    df_by_region_temp['Type'] = df_by_region_temp['Type'].replace(debt_row, debt_row2)
    df_by_region_temp = df_by_region_temp.sort_values(by='Type', key=lambda x: x.map({k:i for i, k in enumerate(order2)}))
    df_by_region_temp['Region'] = x

    df_by_region_main2 = pd.concat([df_by_region_main2, df_by_region_temp], axis = 0)  

df_by_region_main2.reset_index(inplace = True, drop=True)
df_by_region_main2.insert(loc=0, column='Group', value='Liabilities')

df_by_region_main2#[(df_by_region_main2['Type'] == 'Other Investments')]
# ###################### this is for debt ######################

# ###################### this is for caq data ######################

# caq_data = df_temp_col_1[(df_temp_col_1['Type'].isin(df_init[(df_init['group'] == 'Current Account')]['desc'].to_list()[1:]))]
# caq_data_calc = caq_data.groupby(['Type']).sum()
# caq_data_calc.loc['Current Account Balance'] = caq_data_calc.sum()
# caq_data_calc['Region'] = 'Asia-Pacific'
# caq_data_calc.reset_index(inplace=True)
# rename_to = ['Goods Trade', 'Services Trade','Primary Income','Secondary Income',]
# rename_from = ['BoP: Current Account: Goods','BoP: Current Account: Services','BoP: Current Account: Primary Income','BoP: Current Account: Secondary Income',]
# caq_data_calc['Type'] = caq_data_calc['Type'].replace(rename_from, rename_to)
# caq_data_calc = caq_data_calc.sort_values(by='Type', key=lambda x: x.map({k:i for i, k in enumerate(rename_to)}))
# caq_data_calc.reset_index(drop=True, inplace=True)
# caq_data_calc.insert(loc=0, column='Group', value='Current Account')

# ###################### this is for caq data ######################

# df_main = pd.concat([df_by_region_main, df_by_region_main2, caq_data_calc], axis = 0) ## combine
# df_main.reset_index(drop=True, inplace=True)

C:\Users\Admin\AppData\Local\Temp\ipykernel_13864\1850059041.py:21: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_by_region_main2 = pd.concat([df_by_region_main2, df_by_region_temp], axis = 0)


,Group,Type,Region,1H2019,2H2019,1H2020,2H2020,1H2021,2H2021,1H2022,2H2022,1H2023,2H2023,1H2024,2H2024,1H2025
0,Liabilities,Other Investments,Asia-Pacific,67.550889,7.044588e+01,75.553523,80.238407,81.538383,178.054028,1.722052e+02,173.980331,175.386103,177.748953,168.053657,0,0
1,Liabilities,FDI Liabilities,Asia-Pacific,8211.953720,8.274635e+03,8246.766729,9050.226884,9534.195237,9870.240621,9.847562e+03,10042.010960,10185.504408,10531.540784,10645.730902,0,0
2,Liabilities,Financial Derivative Liabilities,Asia-Pacific,256.804296,2.446795e+02,320.835405,350.282109,304.405608,301.816960,4.573018e+02,486.897024,491.133384,422.469582,413.882560,0,0
3,Liabilities,Currency and Deposits Liabilities,Asia-Pacific,2702.868810,2.722472e+03,2750.602404,2917.642670,3070.920631,3084.988095,3.023057e+03,2944.556503,2878.583383,2953.951722,2779.888292,0,0
4,Liabilities,Loans Liabilities,Asia-Pacific,1599.125414,1.601866e+03,1625.557243,1641.335048,1698.341651,1690.597626,1.617004e+03,1615.502697,1663.974501,1654.389858,1632.546800,0,0
5,Liabilities,Trade Credits and Advances Liabilities,Asia-Pacific,725.723748,7.325734e+02,693.431264,742.259260,775.841444,836.245192,8.401963e+02,828.967007,791.944175,820.930619,722.854679,0,0
6,Liabilities,Portfolio Debt Liabilities,Asia-Pacific,1336.032646,1.410516e+03,1431.157814,1662.863329,1782.984059,1901.868720,1.792263e+03,1792.044677,1758.563085,1865.639569,1954.315924,0,0
7,Liabilities,Portfolio Equity Liabilities,Asia-Pacific,3076.208666,3.144753e+03,3073.090542,3788.894039,4135.344193,4001.421249,3.316351e+03,3246.447258,3485.615139,3409.902864,2831.987641,0,0
8,Liabilities,Asia-Pacific Total,Asia-Pacific,17976.268189,1.820194e+04,18216.994925,20233.741746,21383.571206,21865.232490,2.106594e+04,21130.406457,21430.704178,21836.573950,21149.260455,0,0
9,Liabilities,Other Investments,Asian Advance Economies,30.948839,3.302105e+01,37.262841,38.949565,34.738061,53.556382,5.087216e+01,51.191969,52.474942,54.312748,45.104205,0,0


In [175]:
tw_col1 = df_all_annual_2.columns[4:][~df_all_annual_2.columns[4:].isin(df_tw.columns[4:])]
tw_col2 = {x.year:x for x in df_tw.columns[4:]}

for x in tw_col1:
    year = x.year
    if year in list(tw_col2.keys()):
        df_tw[x] = tw_col2[year]
    else:
        df_tw[x] = None
df_tw[df_tw.columns[4:12]][(df_tw['Type'] == 'IIP: Assets')]

# df_tw.loc[:,tw_col1] = None
# df_tw = df_tw[df_tw.columns[:4].to_list() +df_tw.columns[4:].sort_values().to_list()]
# df_tw
# df_temp_col_1.drop(index=tw_index.index.to_list(), inplace=True)
# df_temp_col_1 = pd.concat([df_temp_col_1, tw_index], axis = 0)

# df_temp_col_1.sort_values(by=['Type', 'Region'], inplace=True)
# df_temp_col_1.reset_index(drop=True, inplace=True)
# df_temp_col_1

,2019-12-01 00:00:00,2020-12-01 00:00:00,2021-12-01 00:00:00,2022-12-01 00:00:00,2023-12-01 00:00:00,2019-03-01 00:00:00,2019-06-01 00:00:00,2019-09-01 00:00:00
0,2281141,2519168,2728856,2662530,2909567,2019-12-01,2019-12-01,2019-12-01


In [178]:
df_temp_col_1[(df_temp_col_1['Region'] == 'Taiwan')]

,Type,Region,Unit,Last Update Time,1H2019,2H2019,1H2020,2H2020,1H2021,2H2021,1H2022,2H2022,1H2023,2H2023,1H2024,2H2024,1H2025
16,IIP: Assets,Taiwan,USD mn,2024-06-14,2019-12-01 00:00:00,2281141.0,2020-12-01 00:00:00,2519168.0,2021-12-01 00:00:00,2728856.0,2022-12-01 00:00:00,2662530.0,2023-12-01 00:00:00,2909567.0,NaN,NaN,NaN
35,IIP: Assets: Direct Investment,Taiwan,USD mn,2024-06-14,2019-12-01 00:00:00,368040.0,2020-12-01 00:00:00,400725.0,2021-12-01 00:00:00,450961.0,2022-12-01 00:00:00,450919.0,2023-12-01 00:00:00,500135.0,NaN,NaN,NaN
54,IIP: Assets: Direct Investment: Debt Instruments,Taiwan,USD mn,2024-06-14,2019-12-01 00:00:00,8096.0,2020-12-01 00:00:00,8654.0,2021-12-01 00:00:00,9528.0,2022-12-01 00:00:00,11289.0,2023-12-01 00:00:00,11058.0,NaN,NaN,NaN
73,IIP: Assets: Direct Investment: Equity,Taiwan,USD mn,2024-06-14,2019-12-01 00:00:00,359944.0,2020-12-01 00:00:00,392071.0,2021-12-01 00:00:00,441433.0,2022-12-01 00:00:00,439630.0,2023-12-01 00:00:00,489077.0,NaN,NaN,NaN
92,IIP: Assets: Financial Derivatives & Employee ...,Taiwan,USD mn,2024-06-14,2019-12-01 00:00:00,5949.0,2020-12-01 00:00:00,8957.0,2021-12-01 00:00:00,4658.0,2022-12-01 00:00:00,9027.0,2023-12-01 00:00:00,9773.0,NaN,NaN,NaN
111,IIP: Assets: Official Reserve Assets,Taiwan,USD mn,2024-06-14,2019-12-01 00:00:00,483240.0,2020-12-01 00:00:00,535327.0,2021-12-01 00:00:00,553984.0,2022-12-01 00:00:00,559956.0,2023-12-01 00:00:00,575597.0,NaN,NaN,NaN
130,IIP: Assets: Other Investment,Taiwan,USD mn,2024-06-14,2019-12-01 00:00:00,404911.0,2020-12-01 00:00:00,420507.0,2021-12-01 00:00:00,430940.0,2022-12-01 00:00:00,427681.0,2023-12-01 00:00:00,438227.0,NaN,NaN,NaN
149,IIP: Assets: Other Investment: Currency & Depo...,Taiwan,USD mn,2024-06-14,2019-12-01 00:00:00,208183.0,2020-12-01 00:00:00,225952.0,2021-12-01 00:00:00,229403.0,2022-12-01 00:00:00,230985.0,2023-12-01 00:00:00,238765.0,NaN,NaN,NaN
168,"IIP: Assets: Other Investment: Insurance, Pens...",Taiwan,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
187,IIP: Assets: Other Investment: Loans,Taiwan,USD mn,2024-06-14,2019-12-01 00:00:00,114431.0,2020-12-01 00:00:00,119735.0,2021-12-01 00:00:00,124904.0,2022-12-01 00:00:00,123928.0,2023-12-01 00:00:00,120185.0,NaN,NaN,NaN


In [156]:
tw_col2

[datetime.datetime(2019, 12, 1, 0, 0),
 datetime.datetime(2019, 12, 1, 0, 0),
 datetime.datetime(2019, 12, 1, 0, 0),
 datetime.datetime(2019, 12, 1, 0, 0),
 datetime.datetime(2020, 12, 1, 0, 0),
 datetime.datetime(2020, 12, 1, 0, 0),
 datetime.datetime(2020, 12, 1, 0, 0),
 datetime.datetime(2020, 12, 1, 0, 0),
 datetime.datetime(2021, 12, 1, 0, 0),
 datetime.datetime(2021, 12, 1, 0, 0),
 datetime.datetime(2021, 12, 1, 0, 0),
 datetime.datetime(2021, 12, 1, 0, 0),
 datetime.datetime(2022, 12, 1, 0, 0),
 datetime.datetime(2022, 12, 1, 0, 0),
 datetime.datetime(2022, 12, 1, 0, 0),
 datetime.datetime(2022, 12, 1, 0, 0),
 datetime.datetime(2023, 12, 1, 0, 0),
 datetime.datetime(2023, 12, 1, 0, 0),
 datetime.datetime(2023, 12, 1, 0, 0),
 datetime.datetime(2023, 12, 1, 0, 0)]

In [91]:
tw_index = df_temp_col_1[(df_temp_col_1['Region'] =='Taiwan')].copy()

tw_col1 = [x for x in tw_index.columns[4:] if x[:2] == '1H']
tw_col2 = [x for x in tw_index.columns[4:] if x[:2] == '2H']
for x, y in zip(tw_col1, tw_col2):
    tw_index[x] = tw_index[y]
tw_index.index

Index([ 16,  35,  54,  73,  92, 111, 130, 149, 168, 187, 206, 225, 244, 263,
       282, 301, 320, 339, 358, 377, 396, 415, 434, 453, 472, 491, 510, 529,
       548, 567, 586, 605],
      dtype='int64')

In [74]:
df_temp_col_1[(df_temp_col_1['Region'] =='Taiwan')]

,Type,Region,Unit,Last Update Time,1H2019,2H2019,1H2020,2H2020,1H2021,2H2021,1H2022,2H2022,1H2023,2H2023,1H2024,2H2024,1H2025
16,IIP: Assets,Taiwan,USD mn,2024-06-14,NaN,2281.141,NaN,2519.168,NaN,2728.856,NaN,2662.530,NaN,2909.567,NaN,NaN,NaN
35,IIP: Assets: Direct Investment,Taiwan,USD mn,2024-06-14,NaN,368.040,NaN,400.725,NaN,450.961,NaN,450.919,NaN,500.135,NaN,NaN,NaN
54,IIP: Assets: Direct Investment: Debt Instruments,Taiwan,USD mn,2024-06-14,NaN,8.096,NaN,8.654,NaN,9.528,NaN,11.289,NaN,11.058,NaN,NaN,NaN
73,IIP: Assets: Direct Investment: Equity,Taiwan,USD mn,2024-06-14,NaN,359.944,NaN,392.071,NaN,441.433,NaN,439.630,NaN,489.077,NaN,NaN,NaN
92,IIP: Assets: Financial Derivatives & Employee ...,Taiwan,USD mn,2024-06-14,NaN,5.949,NaN,8.957,NaN,4.658,NaN,9.027,NaN,9.773,NaN,NaN,NaN
111,IIP: Assets: Official Reserve Assets,Taiwan,USD mn,2024-06-14,NaN,483.240,NaN,535.327,NaN,553.984,NaN,559.956,NaN,575.597,NaN,NaN,NaN
130,IIP: Assets: Other Investment,Taiwan,USD mn,2024-06-14,NaN,404.911,NaN,420.507,NaN,430.940,NaN,427.681,NaN,438.227,NaN,NaN,NaN
149,IIP: Assets: Other Investment: Currency & Depo...,Taiwan,USD mn,2024-06-14,NaN,208.183,NaN,225.952,NaN,229.403,NaN,230.985,NaN,238.765,NaN,NaN,NaN
168,"IIP: Assets: Other Investment: Insurance, Pens...",Taiwan,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
187,IIP: Assets: Other Investment: Loans,Taiwan,USD mn,2024-06-14,NaN,114.431,NaN,119.735,NaN,124.904,NaN,123.928,NaN,120.185,NaN,NaN,NaN


In [56]:
df_all_annual_2

,Type,Region,Unit,Last Update Time,2019-03-01 00:00:00,2019-06-01 00:00:00,2019-09-01 00:00:00,2019-12-01 00:00:00,2020-03-01 00:00:00,2020-06-01 00:00:00,...,2023-06-01 00:00:00,2023-09-01 00:00:00,2023-12-01 00:00:00,2024-03-01 00:00:00,2024-06-01 00:00:00,2024-09-01 00:00:00,2024-12-01 00:00:00,2025-03-01 00:00:00,2025-06-01 00:00:00,2025-09-01 00:00:00
0,IIP: Assets,Brunei,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,IIP: Assets,Cambodia,USD mn,2024-10-24,2.417597e+04,2.507544e+04,2.677201e+04,2.862880e+04,2.878970e+04,3.065766e+04,...,3.005370e+04,3.036972e+04,3.360136e+04,3.539776e+04,3.580391e+04,NaN,NaN,NaN,NaN,NaN
2,IIP: Assets,China,USD mn,2024-10-24,7.503297e+06,7.592054e+06,7.613220e+06,7.846418e+06,7.777337e+06,8.014177e+06,...,9.366431e+06,9.303764e+06,9.581683e+06,9.664292e+06,9.792861e+06,NaN,NaN,NaN,NaN,NaN
3,IIP: Assets,Hong Kong SAR (China),USD mn,2024-10-24,5.678545e+06,5.533903e+06,5.438994e+06,5.641767e+06,5.543582e+06,5.706870e+06,...,6.112216e+06,6.123758e+06,6.183809e+06,6.268514e+06,6.470570e+06,NaN,NaN,NaN,NaN,NaN
4,IIP: Assets,India,USD mn,2024-10-24,6.421384e+05,6.625843e+05,6.702019e+05,6.980954e+05,7.168001e+05,7.487695e+05,...,9.407915e+05,9.440507e+05,9.916381e+05,1.026863e+06,1.048228e+06,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
584,IIP: Liabilities: Portfolio Investment: Equity,South Korea,USD mn,2024-09-26,4.697642e+05,4.677524e+05,4.474341e+05,4.975231e+05,3.733060e+05,4.389853e+05,...,5.192382e+05,4.861519e+05,5.655252e+05,5.988869e+05,6.077568e+05,NaN,NaN,NaN,NaN,NaN
585,IIP: Liabilities: Portfolio Investment: Equity,Sri Lanka,USD mn,2020-08-27,7.385835e+02,6.513650e+02,9.751852e+02,1.055937e+03,1.990319e+02,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
586,IIP: Liabilities: Portfolio Investment: Equity,Taiwan,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
587,IIP: Liabilities: Portfolio Investment: Equity,Thailand,USD mn,2024-10-24,1.182496e+05,1.275366e+05,1.200384e+05,1.161480e+05,7.432922e+04,8.907222e+04,...,9.603894e+04,9.113267e+04,9.005005e+04,8.106438e+04,7.595472e+04,NaN,NaN,NaN,NaN,NaN


In [ ]:
pd.concat([df_temp_col_1[df_temp_col_1.columns[:2]][(df_temp_col_1['Type']=='IIP: Assets')], df_temp_col_1[df_temp_col_1.columns[2:]][(df_temp_col_1['Type']=='IIP: Assets')]/1000], axis=1)

,Type,Region,1H2019,2H2019,1H2020,2H2020,1H2021,2H2021,1H2022,2H2022,1H2023,2H2023,1H2024,2H2024,1H2025
0,IIP: Assets,Brunei,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,IIP: Assets,Cambodia,0.049251,0.055401,0.059447,0.067575,0.062060,0.058532,0.057874,0.055339,0.060065,0.063971,0.071202,NaN,NaN
2,IIP: Assets,China,15.095351,15.459637,15.791514,17.172499,19.294797,18.956215,18.734448,18.186327,18.808894,18.885447,19.457153,NaN,NaN
3,IIP: Assets,Hong Kong SAR (China),11.212448,11.080761,11.250453,12.229871,12.965785,12.824832,12.578471,11.957092,12.259264,12.307567,12.739084,NaN,NaN
4,IIP: Assets,India,1.304723,1.368297,1.465570,1.655428,1.751156,1.859408,1.835778,1.748168,1.856972,1.935689,2.075091,NaN,NaN
5,IIP: Assets,Indonesia,0.724324,0.742283,0.750765,0.795300,0.825886,0.862788,0.869213,0.889769,0.931945,0.949373,0.977144,NaN,NaN
6,IIP: Assets,Laos,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,IIP: Assets,Malaysia,0.847109,0.855538,0.871443,0.925038,0.973585,1.005236,0.995476,0.987248,1.029305,1.035196,1.060096,NaN,NaN
8,IIP: Assets,Mongolia,0.013430,0.014119,0.014048,0.015230,0.016772,0.016180,0.013312,0.013112,0.014918,0.016978,0.017907,NaN,NaN
9,IIP: Assets,Myanmar,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [24]:
df_temp_col_1[(df_temp_col_1['Type']=='IIP: Assets')]

,Type,Region,1H2019,2H2019,1H2020,2H2020,1H2021,2H2021,1H2022,2H2022,1H2023,2H2023,1H2024,2H2024,1H2025
0,IIP: Assets,Brunei,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,IIP: Assets,Cambodia,49.251410,55.400812,59.447361,67.574788,62.060306,58.531766,57.874145,55.339257,60.064708,63.971080,71.201661,NaN,NaN
2,IIP: Assets,China,15095.351337,15459.637344,15791.513791,17172.498955,19294.796530,18956.214763,18734.448489,18186.327382,18808.894045,18885.446752,19457.153153,NaN,NaN
3,IIP: Assets,Hong Kong SAR (China),11212.448000,11080.761196,11250.452559,12229.870692,12965.785227,12824.831873,12578.470956,11957.092472,12259.263950,12307.567071,12739.083520,NaN,NaN
4,IIP: Assets,India,1304.722689,1368.297282,1465.569636,1655.427998,1751.156057,1859.408122,1835.778114,1748.168061,1856.971906,1935.688768,2075.091003,NaN,NaN
5,IIP: Assets,Indonesia,724.323817,742.283316,750.764564,795.300483,825.885523,862.788313,869.213043,889.768871,931.945167,949.372809,977.144050,NaN,NaN
6,IIP: Assets,Laos,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,IIP: Assets,Malaysia,847.108568,855.537747,871.443380,925.038211,973.584626,1005.236130,995.475752,987.248425,1029.305168,1035.195587,1060.095770,NaN,NaN
8,IIP: Assets,Mongolia,13.429910,14.118793,14.048275,15.229968,16.772060,16.179695,13.311752,13.111590,14.917649,16.977541,17.906857,NaN,NaN
9,IIP: Assets,Myanmar,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
df_init

,group,short_title,desc
0,FDI Assets,FDI Assets,IIP: Assets: Direct Investment
1,FDI Assets,FDI Equity Assets,IIP: Assets: Direct Investment: Equity
2,FDI Assets,FDI Debt Assets,IIP: Assets: Direct Investment: Debt Instruments
3,Portfolio Assets,Portfolio Assets,IIP: Assets: Portfolio Investment
4,Portfolio Assets,Portfolio Equity Assets,IIP: Assets: Portfolio Investment: Equity
5,Portfolio Assets,Portfolio Debt Assets,IIP: Assets: Portfolio Investment: Debt Securi...
6,Financial Derivative Assets,Financial Derivative Assets,IIP: Assets: Financial Derivatives & Employee ...
7,Other Investment Assets,Other Investment Assets,IIP: Assets: Other Investment
8,Other Investment Assets,OI Equity Assets,IIP: Assets: Other Investment: Other Equity
9,Other Investment Assets,Currency and Deposits Assets,IIP: Assets: Other Investment: Currency & Depo...


In [ ]:
###################### this is for equity ######################

asset_row = df_init.iloc[[0,4,5,6, 8, 9,10,11,12,13,14]]['desc'].to_list()
asset_row2 = ['Direct Investment', 'Portfolio Equity', 'Portfolio Debt','Financial Derivatives', 'Other Equity','Currency & Deposits','Loans','Insurance, Pension & Standardized Guarantee Schemes'
            ,'Trade Credit & Advances','Other Accounts Receivable','Reserve Assets', ]
sum_rows = df_init[(df_init['short_title'].isin(['OI Equity Assets','Insurance and Pension Assets','OI Others Assets',]))]['desc'].to_list()
order = ['Direct Investment','Portfolio Equity','Portfolio Debt','Financial Derivatives','Currency & Deposits','Loans','Trade Credit & Advances','Other Investments','Reserve Assets',]
df_by_region_main = pd.DataFrame(columns=df_temp_col_1.columns)

for x,y in calc_dict.items():
    # print(x,y)
    df_by_region_temp = df_temp_col_1[(df_temp_col_1['Region'].isin(y)) & (df_temp_col_1['Type'].isin(asset_row))]
    df_by_region_temp = df_by_region_temp.groupby(['Type']).sum()
    df_by_region_temp.loc[str(x) + ' Total'] = df_by_region_temp.sum()
    df_by_region_temp.loc['Other Investments'] = df_by_region_temp.loc[sum_rows].sum()
    df_by_region_temp.reset_index(inplace = True)
    df_by_region_temp = df_by_region_temp[(~df_by_region_temp['Type'].isin(sum_rows))] #####  the symbol of '~' use similar like notin, the oposite of isin. this is build in python operator
    df_by_region_temp['Type'] = df_by_region_temp['Type'].replace(asset_row, asset_row2)
    df_by_region_temp = df_by_region_temp.sort_values(by='Type', key=lambda x: x.map({k:i for i, k in enumerate(order)}))
    df_by_region_temp['Region'] = x

    df_by_region_main = pd.concat([df_by_region_main, df_by_region_temp], axis = 0)  


df_by_region_main.reset_index(inplace = True, drop=True)
df_by_region_main.insert(loc=0, column='Group', value='Assets')

###################### this is for equity ######################

###################### this is for debt ######################

debt_row = df_init.iloc[[20,24,25,26,28,29,30,31,32,33,34]]['desc'].to_list()
debt_row2 = ['Direct Investment', 'Portfolio Equity', 'Portfolio Debt','Financial Derivatives', 'Other Equity','Currency & Deposits','Loans','Insurance, Pension & Standardized Guarantee Schemes'
            ,'Trade Credit & Advances','Other Accounts Payable','SDR Liabilities', ]
sum_rows2 = df_init[(df_init['short_title'].isin(['OI Equity Liabilities', 'Insurance and Pension Liabilities', 'OI Others Liabilities', 'SDR Liabilities']))]['desc'].to_list()
order2 = ['Direct Investment','Portfolio Equity','Portfolio Debt','Financial Derivatives','Currency & Deposits','Loans','Trade Credit & Advances','Other Investments']
df_by_region_main2 = pd.DataFrame(columns=df_temp_col_1.columns)

for x,y in calc_dict.items():
    # print(x,y)
    df_by_region_temp = df_temp_col_1[(df_temp_col_1['Region'].isin(y)) & (df_temp_col_1['Type'].isin(debt_row))]
    df_by_region_temp = df_by_region_temp.groupby(['Type']).sum()
    df_by_region_temp.loc[str(x) + ' Total'] = df_by_region_temp.sum()
    df_by_region_temp.loc['Other Investments'] = df_by_region_temp.loc[sum_rows2].sum()
    df_by_region_temp.reset_index(inplace = True)
    df_by_region_temp = df_by_region_temp[(~df_by_region_temp['Type'].isin(sum_rows2))] #####  the symbol of '~' use similar like notin, the oposite of isin. this is build in python operator
    df_by_region_temp['Type'] = df_by_region_temp['Type'].replace(debt_row, debt_row2)
    df_by_region_temp = df_by_region_temp.sort_values(by='Type', key=lambda x: x.map({k:i for i, k in enumerate(order2)}))
    df_by_region_temp['Region'] = x

    df_by_region_main2 = pd.concat([df_by_region_main2, df_by_region_temp], axis = 0)  

df_by_region_main2.reset_index(inplace = True, drop=True)
df_by_region_main2.insert(loc=0, column='Group', value='Liabilities')

###################### this is for debt ######################

###################### this is for caq data ######################

caq_data = df_temp_col_1[(df_temp_col_1['Type'].isin(df_init[(df_init['group'] == 'Current Account')]['desc'].to_list()[1:]))]
caq_data_calc = caq_data.groupby(['Type']).sum()
caq_data_calc.loc['Current Account Balance'] = caq_data_calc.sum()
caq_data_calc['Region'] = 'Asia-Pacific'
caq_data_calc.reset_index(inplace=True)
rename_to = ['Goods Trade', 'Services Trade','Primary Income','Secondary Income',]
rename_from = ['BoP: Current Account: Goods','BoP: Current Account: Services','BoP: Current Account: Primary Income','BoP: Current Account: Secondary Income',]
caq_data_calc['Type'] = caq_data_calc['Type'].replace(rename_from, rename_to)
caq_data_calc = caq_data_calc.sort_values(by='Type', key=lambda x: x.map({k:i for i, k in enumerate(rename_to)}))
caq_data_calc.reset_index(drop=True, inplace=True)
caq_data_calc.insert(loc=0, column='Group', value='Current Account')

###################### this is for caq data ######################

df_main = pd.concat([df_by_region_main, df_by_region_main2, caq_data_calc], axis = 0) ## combine
df_main.reset_index(drop=True, inplace=True)

df_temp_col_1, df_all_annual_2, df_main

In [ ]:
import pandas as pd
import datetime
from pandasql import sqldf

df1 = pd.read_excel(
    r"C:\Users\Admin\OneDrive - The SOUTH-EAST ASIAN CENTRAL BANKS (SEACEN) RESEARCH AND TRAINING\Capital Flow Monitor\SEACEN CFM January 2025\edited files\Data\Section 1 Charts.xlsx",
    sheet_name=3,
    header=1,
)
df1 = df1.iloc[:, :12]
df2 = df1.copy()
df2 = df2.drop([0, 1], axis=0).reset_index(drop=True)
df2 = df2.rename(columns={"Region": "Date"})
date_new = [x for x in df2.Date if isinstance(x, datetime.datetime)]
df2 = df2.iloc[: len(date_new)]
df2 = df2.melt(id_vars="Date", var_name="Region", value_name="Value")
df2["Date"] = pd.to_datetime(df2["Date"]) + pd.offsets.MonthEnd(0)
df2["Year"] = df2["Date"].apply(lambda x: x.date().strftime("%Y"))
df2["Date2"] = df2["Date"].apply(lambda x: x.date().strftime("%Y-%m"))
year_now = datetime.date.today().year
year2 = [str(x) for x in range(2020, year_now + 1)]
latest_year = int(sorted(df2["Year"].unique())[-1])
country = df2["Region"].unique()

df2 = sqldf(
    f'select Date2 as Date, Year, Region, Value, row_number() over(partition by Region, Year order by Date2) as row_num from df2 where Year in ("{'", "'.join(year2)}")'
)

df2 = df2.groupby("Year").apply(
    lambda x: x[x["row_num"].isin([x["row_num"].min(), x["row_num"].max()])]
)
year2 = df2['Year'].unique()
df2.reset_index(drop=True, inplace=True)
df2.fillna(0, inplace=True)
calc_country = dict()

for x in country:
    for y in year2:
        if len(df2[(df2["Region"] == x) & (df2["Year"] == str(y))]) == 1:
            continue
        df_temp = df2[
            (df2["Region"] == x)
            & (df2["Year"] == str(y))
        ]
        nomi = float(
            df_temp[(df_temp["row_num"] == df_temp["row_num"].min())
            ]["Value"].values[0]
        )
        denomi = float(
            df_temp[(df_temp["row_num"] == df_temp["row_num"].max())
            ]["Value"].values[0]
        )
        calc = 0 if nomi == 0 and denomi == 0 else ((nomi / denomi) - 1) * 100
        if calc_country.get(x) == None:
            calc_country.update({x: {y: calc}})
        else:
            calc_country[x].update({y: calc})
df = pd.DataFrame(calc_country).reset_index()
df = df.melt(id_vars="index", var_name="Region", value_name="Value").rename(
    {"index": "Year"}, axis=1
)
region_sort = list(
    df[(df["Year"] == df["Year"].unique()[-1])].sort_values(
        by="Value", ascending=False
    )["Region"]
)
df["Region"] = pd.Categorical(
    df["Region"], categories=region_sort, ordered=True
)
df = df.sort_values(by=["Region", "Year"], ascending=False)

C:\Users\AhmadAizudeen\AppData\Local\Temp\ipykernel_23716\3445644982.py:29: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df2 = df2.groupby("Year").apply(


In [4]:
import pandas as pd
from pandasql import sqldf
import datetime
import numpy as np
import warnings
from arch import arch_model
import requests
from io import BytesIO
warnings.filterwarnings("ignore")

file_url = 'https://www.iif.com/Portals/0/Files/Databases/monthly_em_portfolio_flows_database.xlsx?ver=2025-03-13-063222-833'
response = requests.get(file_url)

if response.headers.get('Content-Type') == 'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet':
    excel_data = BytesIO(response.content)
    df = pd.read_excel(excel_data, sheet_name='Country Flows_nw', header=3)
else:
    print("Failed to download a valid Excel file. The URL might not be a direct link.")

equity = df[df.columns[[0,1,3,5,7,9,11,13,14,16,45,60]]]
equity.rename(columns={'Unnamed: 0': 'Date'}, inplace=True)
equity['Date'].fillna(0, inplace=True)
date_new = [x for x in equity['Date'] if isinstance(x, datetime.date)]
equity = equity[equity['Date'].isin(date_new)]

debt = df[df.columns[[0,2,4,6,8,10,12,15,61]]]
debt.rename(columns={'Unnamed: 0': 'Date'}, inplace=True)
debt['Date'].fillna(0, inplace=True)
date_new = [x for x in debt['Date'] if isinstance(x, datetime.date)]
debt = debt[debt['Date'].isin(date_new)]
debt.rename(columns={k:v for (k,v) in zip(debt.columns, equity.columns.drop(['Taiwan, China', 'Vietnam', 'Sri lanka']))}, inplace=True)

equity.set_index('Date', inplace=True)
debt.set_index('Date', inplace=True)

df_cf = pd.DataFrame(index=debt.index)

df_cf['Portfolio Debt'] = debt.sum(axis=1)/1000
df_cf['Portfolio Equity'] = equity.sum(axis=1)/1000
df_cf.reset_index(inplace=True)

In [8]:
with pd.ExcelWriter("capital_flow.xlsx") as writer:        
    # use to_excel function and specify the sheet_name and index 
    # to store the dataframe in specified sheet
    equity.to_excel(writer, sheet_name="equity")
    debt.to_excel(writer, sheet_name="debt")
    df_cf.to_excel(writer, sheet_name="df_cf")


In [ ]:
import pandas as pd
from datetime import datetime
import os

df_init = pd.read_excel(r"C:\Users\Admin\OneDrive - The SOUTH-EAST ASIAN CENTRAL BANKS (SEACEN) RESEARCH AND TRAINING\Desktop\NLP Project\cfm-nlp\Automation.xlsx", sheet_name='BoP edit')

path_dict = dict()
list_1 = ['IMF BOP Annual.xlsx', 'IMF BOP Quarterly.xlsx']
# directory_path= r'C:\Users\Admin\OneDrive - The SOUTH-EAST ASIAN CENTRAL BANKS (SEACEN) RESEARCH AND TRAINING\Roger and Aizudeen\Country BoP and IIP Data'
directory_path= r'C:\Users\Admin\OneDrive - The SOUTH-EAST ASIAN CENTRAL BANKS (SEACEN) RESEARCH AND TRAINING\Roger and Aizudeen\Country BoP and IIP Data'
for filename in os.listdir(directory_path):
    if filename in ['01 Emerging Market', '02 G7 Countries']:
        continue
    file_path = os.path.join(directory_path, filename)
    list_temp = []
    for filename2 in os.listdir(file_path):
        if any(word in str(filename2) for word in list_1):            
            list_temp.append(filename2)
            path_dict[filename] = list_temp
        else:
            continue

df_path = []
for k, v in path_dict.items():
    df_path.append(os.path.join(directory_path, k, v[0]))
    print(v[0]) ## to check the file name

df_all_annual = pd.DataFrame()
for x, y in enumerate(df_path):
    df = pd.read_excel(y)
    df = df[df[df.columns[0]].isin(df_init['desc'])]
    df_all_annual = pd.concat([df_all_annual, df]) 

df_all_annual.rename(columns={df_all_annual.columns[0]: 'Type'}, inplace=True)

df_all_annual_2 = df_all_annual[df_all_annual.columns[:4].to_list() + [dt for dt in df_all_annual.columns[4:].sort_values(ascending=True).to_list() if dt >= datetime(2019, 1, 1)]]


# concat to the main df
# df_all_annual_2 = pd.concat([df_all_annual_2, sg_df, df_tw], axis=0)

seacen_country = ['Papua New Guinea', 'Vietnam', 'Nepal', 'India', 'Indonesia', 'Laos', 'Sri Lanka', 'Hong Kong SAR (China)', 'Philippines', 'Taiwan', 'Malaysia', 'Mongolia', 'China', 'Cambodia', 'Thailand', 'Singapore', 'South Korea', 'Brunei', 'Myanmar']

for x in df_all_annual_2['Type'].values:
    var1 = list(set(df_all_annual_2[(df_all_annual_2['Type'] == x)]['Region'].values).symmetric_difference(set(seacen_country)))
    if len(var1) > 0:
        for y in var1:
            df_temp1 = pd.DataFrame({"Type": [x], "Region": [y]})
            df_all_annual_2 = pd.concat([df_all_annual_2, df_temp1], axis=0)
    else:
        continue
df_all_annual_2.replace('South Korea', 'Korea', inplace=True)
df_all_annual_2.sort_values(by=['Type', 'Region'], inplace=True)
df_all_annual_2.reset_index(drop=True, inplace=True)

init_dict = df_init.to_dict()
df_dict = dict()
for x in init_dict['desc'].values():
    df_temp = df_all_annual_2[(df_all_annual_2[df_all_annual_2.columns[0]] == x)].reset_index(drop=True)
    df_dict[x] = df_temp



BN IMF BOP Annual.xlsx
KH IMF BOP Annual.xlsx
CH IMF BOP Annual.xlsx
HK IMF BOP Annual.xlsx
IN IMF BOP Annual.xlsx
ID IMF BOP Annual.xlsx
KR IMF BOP Annual.xlsx
LA IMF BOP Annual.xlsx
MY IMF BOP Annual.xlsx
MG IMF BOP Annual.xlsx
NP IMF BOP Annual.xlsx
PG IMF BOP Annual.xlsx
PH IMF BOP Annual.xlsx
SG IMF BOP Annual.xlsx
LK IMF BOP Annual.xlsx
TH IMF BOP Annual.xlsx
VN IMF BOP Annual.xlsx


In [14]:
df_all_annual_2

,Type,Region,Unit,Last Update Time,2019-12-01 00:00:00,2020-12-01 00:00:00,2021-12-01 00:00:00,2022-12-01 00:00:00,2023-12-01 00:00:00,2024-12-01 00:00:00,2025-12-01 00:00:00
0,BoP: Current Account,Brunei,USD mn,2024-10-24,893.758524,513.713115,1570.169313,3264.028471,1944.053357,NaN,NaN
1,BoP: Current Account,Cambodia,USD mn,2025-04-24,-2934.798883,-348.202155,-10892.923222,-7582.000401,552.346251,222.107652,NaN
2,BoP: Current Account,China,USD mn,2025-04-24,102909.876008,248835.643936,352885.754635,443374.257116,263382.353368,423919.214716,NaN
3,BoP: Current Account,Hong Kong SAR (China),USD mn,2025-04-24,21253.125997,24125.744350,43658.556611,36525.097471,32337.983987,52474.997829,NaN
4,BoP: Current Account,India,USD mn,2025-04-24,-29762.864650,32730.048588,-33422.359358,-79050.941951,-31962.287412,-32428.396011,NaN
...,...,...,...,...,...,...,...,...,...,...,...
660,BoP: Financial Account: Portfolio Investment: ...,Singapore,USD mn,2025-03-26,5338.309947,-242.654120,3984.269747,3510.638298,17957.103493,3617.631651,NaN
661,BoP: Financial Account: Portfolio Investment: ...,Sri Lanka,USD mn,2024-05-23,-4.367752,-216.741810,-232.216254,151.173147,8.089240,NaN,NaN
662,BoP: Financial Account: Portfolio Investment: ...,Taiwan,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
663,BoP: Financial Account: Portfolio Investment: ...,Thailand,USD mn,2025-04-24,55.830303,-7868.487119,-1962.658622,6421.230057,-5657.548885,-3395.481689,NaN


In [15]:
st = df_init['short_title'].to_list()
desc = df_init['desc'].to_list()    
st_desc_dict = dict(zip(desc, st))
st_desc_dict

{'BoP: Financial Account: Direct Investment: Assets': 'FDI Assets',
 'BoP: Financial Account: Direct Investment: Assets: Equity': 'FDI Equity Assets',
 'BoP: Financial Account: Direct Investment: Assets: Debt Instruments': 'FDI Debt Assets',
 'BoP: Financial Account: Portfolio Investment: Assets': 'Portfolio Assets',
 'BoP: Financial Account: Portfolio Investment: Assets: Equity': 'Portfolio Equity Assets',
 'BoP: Financial Account: Portfolio Investment: Assets: Debt Securities': 'Portfolio Debt Assets',
 'BoP: Financial Account: Financial Derivatives & Employee Stock Options: Assets': 'Financial Derivative Assets',
 'BoP: Financial Account: Other Investment: Assets': 'Other Investment Assets',
 'BoP: Financial Account: Other Investment: Assets: Other Equity': 'OI Equity Assets',
 'BoP: Financial Account: Other Investment: Assets: Currency & Deposits': 'Currency and Deposits Assets',
 'BoP: Financial Account: Other Investment: Assets: Loans': 'Loans Assets',
 'BoP: Financial Account: O

In [16]:
df_all_annual_2.replace({'Type': st_desc_dict}, inplace=True)
df_all_annual_2.to_excel('section 2 cfm annual.xlsx', index=False)
df_all_annual_2

,Type,Region,Unit,Last Update Time,2019-12-01 00:00:00,2020-12-01 00:00:00,2021-12-01 00:00:00,2022-12-01 00:00:00,2023-12-01 00:00:00,2024-12-01 00:00:00,2025-12-01 00:00:00
0,Current Account,Brunei,USD mn,2024-10-24,893.758524,513.713115,1570.169313,3264.028471,1944.053357,NaN,NaN
1,Current Account,Cambodia,USD mn,2025-04-24,-2934.798883,-348.202155,-10892.923222,-7582.000401,552.346251,222.107652,NaN
2,Current Account,China,USD mn,2025-04-24,102909.876008,248835.643936,352885.754635,443374.257116,263382.353368,423919.214716,NaN
3,Current Account,Hong Kong SAR (China),USD mn,2025-04-24,21253.125997,24125.744350,43658.556611,36525.097471,32337.983987,52474.997829,NaN
4,Current Account,India,USD mn,2025-04-24,-29762.864650,32730.048588,-33422.359358,-79050.941951,-31962.287412,-32428.396011,NaN
...,...,...,...,...,...,...,...,...,...,...,...
660,Portfolio Equity Liabilities,Singapore,USD mn,2025-03-26,5338.309947,-242.654120,3984.269747,3510.638298,17957.103493,3617.631651,NaN
661,Portfolio Equity Liabilities,Sri Lanka,USD mn,2024-05-23,-4.367752,-216.741810,-232.216254,151.173147,8.089240,NaN,NaN
662,Portfolio Equity Liabilities,Taiwan,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
663,Portfolio Equity Liabilities,Thailand,USD mn,2025-04-24,55.830303,-7868.487119,-1962.658622,6421.230057,-5657.548885,-3395.481689,NaN


In [43]:
import pandas as pd
from datetime import datetime
import os

df_init_tw = pd.read_excel(r"C:\Users\AhmadAizudeen\OneDrive - The SOUTH-EAST ASIAN CENTRAL BANKS (SEACEN) RESEARCH AND TRAINING\Desktop\NLP Project\cfm-nlp\Automation.xlsx", sheet_name='Taiwan IIP')
df_tw = pd.read_excel(r"C:\Users\AhmadAizudeen\OneDrive - The SOUTH-EAST ASIAN CENTRAL BANKS (SEACEN) RESEARCH AND TRAINING\Roger and Aizudeen\Country BoP and IIP Data\Chinese Taipei (Taiwan) (TW)\TW CB IIP Annual.xlsx")
df_tw.rename(columns={df_tw.columns[0]: 'Type'}, inplace=True)
df_tw = df_tw[df_tw['Type'].isin(df_init_tw['desc'])]

for x, y in enumerate(df_tw['Type'].to_list()):
    df_tw.iat[x,0] = list(df_init_tw[(df_init_tw['desc'] == y)]['desc2'])[0]

selected_columns = [col for col in df_tw.columns[4:] if pd.to_datetime(col) >= datetime(2019, 1, 1)]
final_columns = list(df_tw.columns[:4]) + selected_columns
df_tw = df_tw[final_columns]
df_tw

,Type,Region,Unit,Last Update Time,2019-12-01 00:00:00,2020-12-01 00:00:00,2021-12-01 00:00:00,2022-12-01 00:00:00,2023-12-01 00:00:00,2024-12-01 00:00:00
0,IIP: Assets,Taiwan,USD mn,2025-06-16,2283565,2521969,2728730,2661984,2886708,3034873
1,IIP: Assets: Direct Investment,Taiwan,USD mn,2025-06-16,368040,400725,450961,450919,500082,567278
2,IIP: Assets: Direct Investment: Equity,Taiwan,USD mn,2025-06-16,359944,392071,441433,439630,489024,555832
3,IIP: Assets: Direct Investment: Debt Instruments,Taiwan,USD mn,2025-06-16,8096,8654,9528,11289,11058,11446
4,IIP: Assets: Portfolio Investment,Taiwan,USD mn,2025-06-16,1021425,1156453,1288371,1214980,1365172,1457685
5,IIP: Assets: Portfolio Investment: Equity,Taiwan,USD mn,2025-06-16,288868,324537,389311,319376,376016,421722
11,IIP: Assets: Portfolio Investment: Debt Securi...,Taiwan,USD mn,2025-06-16,732557,831916,899060,895604,989156,1035963
17,IIP: Assets: Financial Derivatives & Employee ...,Taiwan,USD mn,2025-06-16,5949,8957,4658,9027,9773,8284
23,IIP: Assets: Other Investment,Taiwan,USD mn,2025-06-16,404911,420507,430756,427102,436084,420215
24,IIP: Assets: Other Investment: Other Equity,Taiwan,USD mn,2025-06-16,80,83,81,77,78,75


In [40]:
# nepal = df_all_annual_2[(df_all_annual_2['Region'] == 'Nepal')]

import pandas as pd 

df_np = pd.read_excel(r"C:\Users\AhmadAizudeen\OneDrive - The SOUTH-EAST ASIAN CENTRAL BANKS (SEACEN) RESEARCH AND TRAINING\Roger and Aizudeen\Country BoP and IIP Data\Nepal (NP)\NP IMF IIP Quarterly.xlsx")
df_np = df_np[df_np[df_np.columns[0]].isin(df_init['desc'].to_list())]
df_np.rename(columns={df_np.columns[0]: 'Type'}, inplace=True)
df_np.replace({'Type': {k:v for (k,v) in zip(df_init['desc'].to_list(), df_init['short_title'].to_list())}}, inplace=True)

df_np

,Type,Region,Unit,Last Update Time,2000-03-01 00:00:00,2000-06-01 00:00:00,2000-09-01 00:00:00,2000-12-01 00:00:00,2001-03-01 00:00:00,2001-06-01 00:00:00,...,2022-03-01 00:00:00,2022-06-01 00:00:00,2022-09-01 00:00:00,2022-12-01 00:00:00,2023-03-01 00:00:00,2023-06-01 00:00:00,2023-09-01 00:00:00,2023-12-01 00:00:00,2024-03-01 00:00:00,2024-06-01 00:00:00
0,International Investment Assets,Nepal,USD mn,2024-10-24,NaN,NaN,NaN,NaN,NaN,NaN,...,10504.215152,10655.073413,10577.172777,10848.069071,11711.314479,12591.034436,13186.697835,14598.910111,15412.905491,16283.241371
1,Other Investment Assets,Nepal,USD mn,2024-10-24,NaN,NaN,NaN,NaN,NaN,NaN,...,1438.335165,1489.469208,1255.348491,1168.362006,1286.566258,1631.803480,1722.485865,1821.857564,1864.898936,1640.358107
2,OI Equity Assets,Nepal,USD mn,2024-10-24,NaN,NaN,NaN,NaN,NaN,NaN,...,97.378091,102.589193,95.140538,91.899355,93.459401,95.126255,97.464833,100.597751,98.903338,100.310165
3,Currency and Deposits Assets,Nepal,USD mn,2024-10-24,NaN,NaN,NaN,NaN,NaN,NaN,...,632.199674,689.829046,469.786375,425.212888,511.581380,517.829285,629.059124,573.332460,604.169012,458.085806
6,Loans Assets,Nepal,USD mn,2024-10-24,NaN,NaN,NaN,NaN,NaN,NaN,...,7.231890,8.204309,1.473172,1.487721,8.752462,11.939345,14.152609,16.816904,18.739147,21.491652
9,Trade Credits and Advances Assets,Nepal,USD mn,2024-10-24,NaN,NaN,NaN,NaN,NaN,NaN,...,143.833023,114.172540,31.216590,-87.963824,-23.813265,58.648837,46.230709,87.788548,68.952018,90.895459
14,OI Others Assets,Nepal,USD mn,2024-10-24,NaN,NaN,NaN,NaN,NaN,NaN,...,557.692486,574.674120,657.731816,737.725866,696.586281,948.259758,935.578591,1043.321901,1074.135421,969.575025
17,Official Reserve Assets,Nepal,USD mn,2024-10-24,NaN,NaN,NaN,NaN,NaN,NaN,...,9065.879987,9165.604204,9321.824286,9679.707065,10424.748222,10959.230956,11464.211970,12777.052547,13548.006556,14642.883264
22,International Investment Liabilities,Nepal,USD mn,2024-10-24,NaN,NaN,NaN,NaN,NaN,NaN,...,12484.462084,13173.276661,12187.395519,12510.531338,13439.401863,13548.665463,13264.378666,13711.249514,14152.055516,14566.998655
23,FDI Liabilities,Nepal,USD mn,2024-10-24,NaN,NaN,NaN,NaN,NaN,NaN,...,2010.537361,2117.007304,2073.471628,2009.086812,2031.587017,2057.634280,2245.436376,2259.062515,2280.113716,2273.319648


In [42]:
df_np.to_excel('nepal_iip_quarterly.xlsx', index=False)

In [21]:
df_all_annual[(df_all_annual['Region'] == 'Nepal')]

,Type,Region,Unit,Last Update Time,2000-03-01 00:00:00,2000-06-01 00:00:00,2000-09-01 00:00:00,2000-12-01 00:00:00,2001-03-01 00:00:00,2001-06-01 00:00:00,...,2023-12-01 00:00:00,2024-03-01 00:00:00,2024-06-01 00:00:00,2024-09-01 00:00:00,2024-12-01 00:00:00,2025-03-01 00:00:00,2025-06-01 00:00:00,2025-09-01 00:00:00,2025-12-01 00:00:00,2026-03-01 00:00:00
0,IIP: Assets,Nepal,USD mn,2023-11-16,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,IIP: Assets: Other Investment,Nepal,USD mn,2023-11-16,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,IIP: Assets: Other Investment: Other Equity,Nepal,USD mn,2023-11-16,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,IIP: Assets: Other Investment: Currency & Depo...,Nepal,USD mn,2023-11-16,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,IIP: Assets: Other Investment: Loans,Nepal,USD mn,2023-11-16,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,IIP: Assets: Other Investment: Trade Credit & ...,Nepal,USD mn,2023-11-16,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14,IIP: Assets: Other Investment: Other Accounts ...,Nepal,USD mn,2023-11-16,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17,IIP: Assets: Official Reserve Assets,Nepal,USD mn,2023-11-16,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
22,IIP: Liabilities,Nepal,USD mn,2023-11-16,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
23,IIP: Liabilities: Direct Investment,Nepal,USD mn,2023-11-16,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [20]:
from data_section_2 import bop_quarterly, bop_annual, iip_annual, iip_quarterly
# y,z = iip_annual() #taiwan
# z - Group	Type	Region	Unit	Last Update Time	2019	2020	2021	2022	2023	2024	2025
# y - Type	Region	Unit	Last Update Time	2019	2020	2021	2022	2023	2024	2025
y,z = bop_annual()
# z - Group	Type	Region	Unit	Last Update Time	2019	2020	2021	2022	2023	2024	2025
# y - Type	Region	Unit	Last Update Time	2019	2020	2021	2022	2023	2024	2025
# x, y,z = iip_quarterly() # taiwan
# z - Group	Type	Region	1H2019	2H2019	1H2020	2H2020	1H2021	2H2021	1H2022	2H2022	1H2023	2H2023	1H2024	2H2024	1H2025	2H2025
# y - Type	Region	Unit	Last Update Time	2019-03-01 00:00:00	2019-06-01 00:00:00	2019-09-01 00:00:00	2019-12-01 00:00:00	2020-03-01 00:00:00	2020-06-01 00:00:00	...	2023-12-01 00:00:00	2024-03-01 00:00:00	2024-06-01 00:00:00	2024-09-01 00:00:00	2024-12-01 00:00:00	2025-03-01 00:00:00	2025-06-01 00:00:00	2025-09-01 00:00:00	2025-12-01 00:00:00	2026-03-01 00:00:00
# x - Type	Region	1H2019	2H2019	1H2020	2H2020	1H2021	2H2021	1H2022	2H2022	1H2023	2H2023	1H2024	2H2024	1H2025	2H2025
# x, y,z = bop_quarterly()
# z - Group	Type	Region	1H2019	2H2019	1H2020	2H2020	1H2021	2H2021	1H2022	2H2022	1H2023	2H2023	1H2024	2H2024	1H2025	2H2025
# y - Type	Region	Unit	Last Update Time	2019-03-01 00:00:00	2019-06-01 00:00:00	2019-09-01 00:00:00	2019-12-01 00:00:00	2020-03-01 00:00:00	2020-06-01 00:00:00	...	2023-12-01 00:00:00	2024-03-01 00:00:00	2024-06-01 00:00:00	2024-09-01 00:00:00	2024-12-01 00:00:00	2025-03-01 00:00:00	2025-06-01 00:00:00	2025-09-01 00:00:00	2025-12-01 00:00:00	2026-03-01 00:00:00
# x - Type	Region	1H2019	2H2019	1H2020	2H2020	1H2021	2H2021	1H2022	2H2022	1H2023	2H2023	1H2024	2H2024	1H2025	2H2025
z.head()

BN IMF BOP Annual.xlsx
KH IMF BOP Annual.xlsx
CH IMF BOP Annual.xlsx
HK IMF BOP Annual.xlsx
IN IMF BOP Annual.xlsx
ID IMF BOP Annual.xlsx
KR IMF BOP Annual.xlsx
LA IMF BOP Annual.xlsx
MY IMF BOP Annual.xlsx
MG IMF BOP Annual.xlsx
NP IMF BOP Annual.xlsx
PG IMF BOP Annual.xlsx
PH IMF BOP Annual.xlsx
SG IMF BOP Annual.xlsx
LK IMF BOP Annual.xlsx
TH IMF BOP Annual.xlsx
VN IMF BOP Annual.xlsx


,Group,Type,Region,Unit,Last Update Time,2019,2020,2021,2022,2023,2024,2025
0,Assets,Direct Investment,Asia-Pacific,NaN,NaN,296187.006972,336568.481224,413974.272625,454920.378726,430683.863581,395024.247778,0.0
1,Assets,Portfolio Equity,Asia-Pacific,NaN,NaN,147974.625416,260730.300491,292459.542441,169902.799292,135807.022096,258720.440022,0.0
2,Assets,Portfolio Debt,Asia-Pacific,NaN,NaN,164483.395891,61947.482758,127514.325064,215600.167225,203967.567457,335264.051225,0.0
3,Assets,Financial Derivatives,Asia-Pacific,NaN,NaN,-50129.852234,-60138.823070,-93848.854390,-93372.656538,-75946.867195,-75647.468857,0.0
4,Assets,Currency & Deposits,Asia-Pacific,NaN,NaN,140062.145184,308219.161494,322344.900194,154787.245883,137438.768620,106657.383443,0.0


In [21]:
y.head()

,Type,Region,Unit,Last Update Time,2019,2020,2021,2022,2023,2024,2025
0,Current Account,Brunei,USD mn,2024-10-24,893.758524,513.713115,1570.169313,3264.028471,1944.053357,NaN,NaN
1,Current Account,Cambodia,USD mn,2025-04-24,-2934.798883,-348.202155,-10892.923222,-7582.000401,552.346251,222.107652,NaN
2,Current Account,China,USD mn,2025-04-24,102909.876008,248835.643936,352885.754635,443374.257116,263382.353368,423919.214716,NaN
3,Current Account,Hong Kong SAR (China),USD mn,2025-04-24,21253.125997,24125.744350,43658.556611,36525.097471,32337.983987,52474.997829,NaN
4,Current Account,India,USD mn,2025-04-24,-29762.864650,32730.048588,-33422.359358,-79050.941951,-31962.287412,-32428.396011,NaN


In [3]:
from data_section_2 import iip_annual, iip_quarterly, bop_annual, bop_quarterly, main
import pandas as pd

quarterly_type_half, quarterly_type_quarter, quarterly_region_half, annual_type_full, annual_region_full = main(year_use=2022)

KH IMF IIP Quarterly.xlsx
CH IMF IIP Quarterly.xlsx
HK IMF IIP Quarterly.xlsx
IN IMF IIP Quarterly.xlsx
ID IMF IIP Quarterly.xlsx
KR IMF IIP Quarterly.xlsx
MY IMF IIP Quarterly.xlsx
MG IMF IIP Quarterly.xlsx
NP IMF IIP Quarterly.xlsx
PH IMF IIP Quarterly.xlsx
SG IMF IIP Quarterly.xlsx
LK IMF IIP Quarterly.xlsx
TH IMF IIP Quarterly.xlsx
KH IMF IIP Annual.xlsx
CH IMF IIP Annual.xlsx
HK IMF IIP Annual.xlsx
IN IMF IIP Annual.xlsx
ID IMF IIP Annual.xlsx
KR IMF IIP Annual.xlsx
MY IMF IIP Annual.xlsx
MG IMF IIP Annual.xlsx
NP IMF IIP Annual.xlsx
PH IMF IIP Annual.xlsx
SG IMF IIP Annual.xlsx
LK IMF IIP Annual.xlsx
TH IMF IIP Annual.xlsx
BN IMF BOP Quarterly.xlsx
KH IMF BOP Quarterly.xlsx
CH IMF BOP Quarterly.xlsx
HK IMF BOP Quarterly.xlsx
IN IMF BOP Quarterly.xlsx
ID IMF BOP Quarterly.xlsx
KR IMF BOP Quarterly.xlsx
LA IMF BOP Quarterly.xlsx
MY IMF BOP Quarterly.xlsx
MG IMF BOP Quarterly.xlsx
NP IMF BOP Quarterly.xlsx
PG IMF BOP Quarterly.xlsx
PH IMF BOP Quarterly.xlsx
SG IMF BOP Quarterly.xlsx

In [4]:
quarterly_type_half

,Group2,Type,Region,1H2022,2H2022,1H2023,2H2023,1H2024,2H2024,1H2025,2H2025
0,IIP Quarterly,International Investment Assets,Brunei,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,IIP Quarterly,International Investment Assets,Cambodia,2.869125e+04,2.761402e+04,3.026128e+04,3.335211e+04,3.418877e+04,3.756138e+04,NaN,NaN
2,IIP Quarterly,International Investment Assets,China,9.278640e+06,9.215533e+06,9.387984e+06,9.624811e+06,9.853377e+06,1.021671e+07,NaN,NaN
3,IIP Quarterly,International Investment Assets,Hong Kong SAR (China),6.225264e+06,6.071108e+06,6.125528e+06,6.179129e+06,6.470570e+06,6.763809e+06,NaN,NaN
4,IIP Quarterly,International Investment Assets,India,9.029065e+05,8.876530e+05,9.464027e+05,9.971768e+05,1.048549e+06,1.074639e+06,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
660,BoP Quarterly,Portfolio Equity Liabilities,Singapore,6.676622e+00,9.961362e+00,-5.323875e+00,7.408636e+00,7.869445e+00,1.219132e+01,NaN,NaN
661,BoP Quarterly,Portfolio Equity Liabilities,Sri Lanka,1.389181e-01,1.225502e-02,4.789605e-03,3.299634e-03,NaN,NaN,NaN,NaN
662,BoP Quarterly,Portfolio Equity Liabilities,Taiwan,-3.031800e+01,-1.457600e+01,1.310100e+01,-8.028000e+00,4.940000e+00,-2.383300e+01,NaN,NaN
663,BoP Quarterly,Portfolio Equity Liabilities,Thailand,4.185959e+00,2.235271e+00,-3.384421e+00,-2.273128e+00,-1.231803e+00,-2.163679e+00,NaN,NaN


In [7]:
df1_1.insert(loc=0, column='Group2', value='IIP Quarterly')

In [11]:
annual_region_full

,Group2,Group,Type,Region,Unit,Last Update Time,2019,2020,2021,2022,2023,2024,2025
0,IIP Annual,Assets,Direct Investment,Asia-Pacific,NaN,NaN,6.419955e+06,7.186901e+06,7.713668e+06,7.817091e+06,8.278072e+06,8.806465e+06,0.0
1,IIP Annual,Assets,Portfolio Equity,Asia-Pacific,NaN,NaN,2.785477e+06,3.493191e+06,3.699286e+06,3.265167e+06,3.372352e+06,3.966764e+06,0.0
2,IIP Annual,Assets,Portfolio Debt,Asia-Pacific,NaN,NaN,2.466393e+06,2.700601e+06,2.748150e+06,2.826693e+06,3.098729e+06,3.463662e+06,0.0
3,IIP Annual,Assets,Financial Derivatives,Asia-Pacific,NaN,NaN,2.128914e+05,2.995883e+05,2.694520e+05,3.984054e+05,3.450215e+05,4.018107e+05,0.0
4,IIP Annual,Assets,Currency & Deposits,Asia-Pacific,NaN,NaN,2.321637e+06,2.576051e+06,2.742064e+06,2.843342e+06,2.996575e+06,3.028307e+06,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
114,BoP Annual,Current Account,Goods Trade,Asia-Pacific,NaN,NaN,4.678647e+05,7.601406e+05,7.361287e+05,7.056027e+05,6.948796e+05,8.760599e+05,0.0
115,BoP Annual,Current Account,Services Trade,Asia-Pacific,NaN,NaN,-1.416538e+05,-9.640788e+04,-1.889218e+04,3.994029e+04,-5.442583e+04,-3.157387e+04,0.0
116,BoP Annual,Current Account,Primary Income,Asia-Pacific,NaN,NaN,-1.638041e+05,-2.248721e+05,-2.637768e+05,-3.416127e+05,-3.023975e+05,-2.818824e+05,0.0
117,BoP Annual,Current Account,Secondary Income,Asia-Pacific,NaN,NaN,1.294579e+05,1.345745e+05,1.469487e+05,1.668827e+05,1.725149e+05,1.931535e+05,0.0
